# Computação Gráfica – Projeto 2

### Dupla:

#### Maicon Chaves Marques - 14593530
#### Arthur Trottmann Ramos - 14681052

## Código

### Importação de Bibliotecas

In [295]:
%pip install glfw pyopengl pyglm numpy pillow

import glfw
from OpenGL.GL import *
import numpy as np
import glm
import math
from numpy import random
from PIL import Image
import os

from Shaders.shader_s import Shader

Note: you may need to restart the kernel to use updated packages.


### Inicialização da Janela

In [296]:
glfw.init()
glfw.window_hint(glfw.VISIBLE, glfw.FALSE)

altura = 700
largura = 700

window = glfw.create_window(largura, altura, "Programa", None, None)

if (window == None):
    print("Failed to create GLFW window")
    glfwTerminate()
    
glfw.make_context_current(window)


### Construção e Linkagem de Shaders

In [297]:
ourShader = Shader("Shaders/vertex_shader.vs", "Shaders/fragment_shader.fs")
ourShader.use()

program = ourShader.getProgram()

### Carregamento de Modelos (Vértices e Texturas) via Arquivos

In [298]:
glEnable(GL_TEXTURE_2D)
glHint(GL_LINE_SMOOTH_HINT, GL_DONT_CARE)
glEnable(GL_BLEND)
glBlendFunc(GL_SRC_ALPHA, GL_ONE_MINUS_SRC_ALPHA)
glEnable(GL_LINE_SMOOTH)


global vertices_list
global textures_coord_list
vertices_list = []
textures_coord_list = []
    
global number_textures
number_textures = 0


def load_model_from_file(filename):
    '''
    Leitura do .obj para vértices, faces e vértices de texturas
    '''

    vertices = []
    texture_coords = []
    faces = []

    material = None
    object_name = 'default'

    for line in open(filename, "r"):
        if line.startswith('#'): continue
        values = line.split()
        if not values: continue

        # Coordenadas dos Vértices do Objeto
        if values[0] == 'v':
            vertices.append(values[1:4])

        # Coordenada da Textura
        elif values[0] == 'vt':
            texture_coords.append(values[1:3])

        # Nome do objeto
        elif values[0] == 'o':
            object_name = values[1]

        # Textura da Face
        elif values[0] in ('usemtl', 'usemat'):
            material = values[1]
        
        # Face
        elif values[0] == 'f':
            face = []
            face_texture = []
            for v in values[1:]:
                w = v.split('/')
                face.append(int(w[0]))
                if len(w) >= 2 and len(w[1]) > 0:
                    face_texture.append(int(w[1]))
                else:
                    face_texture.append(0)

            faces.append((face, face_texture, material, object_name))

    model = {}
    model['vertices'] = vertices
    model['texture'] = texture_coords
    model['faces'] = faces

    return model


def load_texture_from_file(texture_id, img_textura):
    '''
    Upload da textura para GPU
    '''

    glBindTexture(GL_TEXTURE_2D, texture_id)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_S, GL_REPEAT)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_T, GL_REPEAT)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)

    img = Image.open(img_textura)
    img_width = img.size[0]
    img_height = img.size[1]
    image_data = img.tobytes("raw", "RGB", 0, -1)

    glTexImage2D(GL_TEXTURE_2D, 0, GL_RGB, img_width, img_height, 0, GL_RGB, GL_UNSIGNED_BYTE, image_data)


def circular_sliding_window_of_three(vertices_arr):
    '''
    Converte uma face em triângulos
    '''

    if len(vertices_arr) == 3:
        return vertices_arr
    
    circular_arr = vertices_arr + [vertices_arr[0]]
    result = []

    for i in range(len(circular_arr) - 2):
        result.extend(circular_arr[i:i+3])

    return result

def get_sub_object_center(sub):
    verts = vertices_list[sub.start_vertex : sub.start_vertex + sub.quantity_vertex]
    
    xs = [float(v[0]) for v in verts]
    ys = [float(v[1]) for v in verts]
    zs = [float(v[2]) for v in verts]

    # Usando o centro geométrico exato (Bounding Box) em vez da média
    cx = (max(xs) + min(xs)) / 2.0
    cy = (max(ys) + min(ys)) / 2.0
    cz = (max(zs) + min(zs)) / 2.0

    return (cx, cy, cz)

def get_object_center(obj):
    verts = vertices_list[obj.subobjects[0].start_vertex : 
                          obj.subobjects[-1].start_vertex + obj.subobjects[-1].quantity_vertex]
    
    xs = [float(v[0]) for v in verts]
    ys = [float(v[1]) for v in verts]
    zs = [float(v[2]) for v in verts]

    # Mesma correção para o centro global do objeto
    cx = (max(xs) + min(xs)) / 2.0
    cy = (max(ys) + min(ys)) / 2.0
    cz = (max(zs) + min(zs)) / 2.0

    return (cx, cy, cz)

def get_object_bottom_y(obj):
    verts = vertices_list[obj.subobjects[0].start_vertex : 
                          obj.subobjects[-1].start_vertex + obj.subobjects[-1].quantity_vertex]
    
    ys = [float(v[1]) for v in verts]

    return max(ys)

### Classes de Objetos e Subobjetos

In [299]:
class SubObjeto:
    def __init__(self, sub_object_name, material, start_vertex, quantity_vertex):
        self.name = sub_object_name
        self.material_name = material
        self.start_vertex = start_vertex
        self.quantity_vertex = quantity_vertex
    
        self.texture_id = None
        self.color = [1.0, 1.0, 1.0, 1.0]

        # Centro
        self.pivot_x, self.pivot_y, self.pivot_z = 0, 0, 0

        self.t_x, self.t_y, self.t_z = 0, 0, 0
        self.scale_x, self.scale_y, self.scale_z = 1, 1, 1
        self.angle_x, self.angle_y, self.angle_z = 0, 0, 0

    def set_color(self, r, g, b, a):
        self.color = [r, g, b, a]

    def set_transformation_subobject(self, t_x, t_y, t_z, angle_x, angle_y, angle_z, scale_x, scale_y, scale_z):
        self.t_x, self.t_y, self.t_z = t_x, t_y, t_z
        self.angle_x, self.angle_y, self.angle_z = angle_x, angle_y, angle_z
        self.scale_x, self.scale_y, self.scale_z = scale_x, scale_y, scale_z
        

class Objeto:
    def __init__(self, obj_file, textures_dict):
        self.name = obj_file.split('/')[-1].replace('.obj', '')
        self.subobjects = []

        # Centro
        self.pivot_x, self.pivot_y, self.pivot_z = 0, 0, 0

        # Transformações globais do objeto
        self.t_x, self.t_y, self.t_z = 0, 0, 0
        self.scale_x, self.scale_y, self.scale_z = 1, 1, 1
        self.angle_x, self.angle_y, self.angle_z = 0, 0, 0

        self.load(obj_file, textures_dict)


    def set_transformations(self, t_x, t_y, t_z, angle_x, angle_y, angle_z, scale_x, scale_y, scale_z):
        self.t_x, self.t_y, self.t_z = t_x, t_y, t_z
        self.angle_x, self.angle_y, self.angle_z = angle_x, angle_y, angle_z
        self.scale_x, self.scale_y, self.scale_z = scale_x, scale_y, scale_z

    def scale_object(self, add_x, add_y, add_z):
        self.scale_x += add_x
        self.scale_y += add_y
        self.scale_z += add_z

    def load(self, obj_file, textures_dict):
        model = load_model_from_file(obj_file)

        faces_per_object = {}

        for face in model['faces']:
            object_name = face[3]
            if object_name not in faces_per_object:
                faces_per_object[object_name] = []
            faces_per_object[object_name].append(face)

        for object_name, faces in faces_per_object.items():
            start_vertex = len(vertices_list)

            material = faces[0][2]

            for face in faces:
                for vertex_id in circular_sliding_window_of_three(face[0]):
                    vertices_list.append(model['vertices'][vertex_id - 1])
                for texture_id in circular_sliding_window_of_three(face[1]):
                    textures_coord_list.append(model['texture'][texture_id - 1])

            quantidade = len(vertices_list) - start_vertex

            sub = SubObjeto(object_name, material, start_vertex, quantidade)

            # Carrega a textura correspondente, se existir
            if material in textures_dict:
                global number_textures
                load_texture_from_file(number_textures, textures_dict[material])
                sub.texture_id = number_textures
                number_textures += 1

            self.subobjects.append(sub)


    def draw(self, program):
        mat_to_origin   = model(0, 0, 0, -self.pivot_x, -self.pivot_y, -self.pivot_z, 1, 1, 1)
        mat_transform   = model(self.angle_x, self.angle_y, self.angle_z, self.t_x, self.t_y, self.t_z, self.scale_x, self.scale_y, self.scale_z)
        mat_from_origin = model(0, 0, 0, self.pivot_x, self.pivot_y, self.pivot_z, 1, 1, 1)

        mat_global = mat_from_origin @ mat_transform @ mat_to_origin

        loc_model     = glGetUniformLocation(program, "model")
        loc_color     = glGetUniformLocation(program, "color")
        loc_use_color = glGetUniformLocation(program, "use_color")

        for sub in self.subobjects:
            # compõe transformação local do sub em cima da global do objeto
            if any([sub.t_x, sub.t_y, sub.t_z,
                    sub.angle_x, sub.angle_y, sub.angle_z,
                    sub.scale_x != 1, sub.scale_y != 1, sub.scale_z != 1]):
                
                sub_to_origin  =  model(0, 0, 0, -sub.pivot_x, -sub.pivot_y, -sub.pivot_z, 1, 1, 1)
                sub_transform  =  model(sub.angle_x, sub.angle_y, sub.angle_z, sub.t_x, sub.t_y, sub.t_z, sub.scale_x, sub.scale_y, sub.scale_z)
                sub_from_origin = model(0, 0, 0, sub.pivot_x, sub.pivot_y, sub.pivot_z, 1, 1, 1)
                
                mat_local = sub_from_origin @ sub_transform @ sub_to_origin
                mat_final = mat_global @ mat_local
            else:
                mat_final = mat_global

            glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_final)

            if sub.texture_id is not None:
                glBindTexture(GL_TEXTURE_2D, sub.texture_id)
                glUniform1i(loc_use_color, 0)
            else:
                glUniform1i(loc_use_color, 1)
                glUniform4f(loc_color, *sub.color)

            glDrawArrays(GL_TRIANGLES, sub.start_vertex, sub.quantity_vertex)

### Carregamento dos Objetos

In [300]:
Objects = []

# =========================
# DEFINIÇÃO DOS OBJETOS
# =========================

Barril = Objeto(
    obj_file='Objetos/Barril/barril.obj',
    textures_dict={
        'Oak_Wood_Floor': 'Objetos/Barril/Oak_Wood_Floor_baseColor.jpeg',
        'Rust_3.001': 'Objetos/Barril/Rust_3.001_baseColor.jpeg'
    }
)

Cacto01 = Objeto(
    obj_file='Objetos/Cacto01/cacto01.obj',
    textures_dict={
        'cactus_0Mat': 'Objetos/Cacto01/cactus_0Mat_baseColor.png',
        'cactus_blend_1Mat': 'Objetos/Cacto01/cactus_blend_1Mat_baseColor.png',
        'prickly_pear_4Mat': 'Objetos/Cacto01/prickly_pear_4Mat_baseColor.png',
        'prickly_pear_blend_3Mat': 'Objetos/Cacto01/prickly_pear_blend_3Mat_baseColor.png',
        'prickly_pear_flower_9Mat': 'Objetos/Cacto01/prickly_pear_flower_9Mat_baseColor.png',
        'prickly_pear_stem_5Mat': 'Objetos/Cacto01/prickly_pear_stem_5Mat_baseColor.png',
        'saguaro_flower_7Mat': 'Objetos/Cacto01/saguaro_flower_7Mat_baseColor.png',
        'saguaro_stem_2Mat': 'Objetos/Cacto01/saguaro_stem_2Mat_baseColor.png',
        'spines_01_6Mat': 'Objetos/Cacto01/spines_01_6Mat_baseColor.png',
        'spines_02_8Mat': 'Objetos/Cacto01/spines_02_8Mat_baseColor.png',
    }
)

Cacto02 = Objeto(
    obj_file='Objetos/Cacto02/cacto02.obj',
    textures_dict={
        'agave_2Mat': 'Objetos/Cacto02/agave_2Mat_baseColor.png',
        'bark_02_0Mat': 'Objetos/Cacto02/bark_02_0Mat_baseColor.png',
        'bark_1Mat': 'Objetos/Cacto02/bark_1Mat_baseColor.png',
        'cholla_01_9Mat': 'Objetos/Cacto02/cholla_01_9Mat_baseColor.png',
        'cholla_02_8Mat': 'Objetos/Cacto02/cholla_02_8Mat_baseColor.png',
        'creosote_branch_02_7Mat': 'Objetos/Cacto02/creosote_branch_02_7Mat_baseColor.png',
        'ocotillo_branch_01_5Mat': 'Objetos/Cacto02/ocotillo_branch_01_5Mat_baseColor.png',
        'ocotillo_branch_02_6Mat': 'Objetos/Cacto02/ocotillo_branch_02_6Mat_baseColor.png',
        'yacca_leaf_01_3Mat': 'Objetos/Cacto02/yacca_leaf_01_3Mat_baseColor.png',
        'yacca_leaf_02_4Mat': 'Objetos/Cacto02/yacca_leaf_02_4Mat_baseColor.png',
    }
)

Cadeira = Objeto(
    obj_file='Objetos/Cadeira/cadeira.obj',
    textures_dict={'Material.002': 'Objetos/Cadeira/Chair_albedo.png'}
)

Cama = Objeto(
    obj_file='Objetos/Cama/cama.obj',
    textures_dict={'Material.001': 'Objetos/Cama/internals_albedo.png'}
)

Casa = Objeto(
    obj_file='Objetos/Casa/casa.obj',
    textures_dict={
        'Ablakok': 'Objetos/Casa/Ablakok_baseColor.jpeg',
        'AblakosKulso': 'Objetos/Casa/AblakosKulso_baseColor.jpeg',
        'Ajto': 'Objetos/Casa/FentiAjto_baseColor.jpeg',
        'Bejarat': 'Objetos/Casa/Bejarat_baseColor.jpeg',
        'EmeletTalaj': 'Objetos/Casa/EmeletTalaj_baseColor.jpeg',
        'FentiAjto': 'Objetos/Casa/FentiAjto_baseColor.jpeg',
        'Korlat': 'Objetos/Casa/Korlat_baseColor.jpeg',
        'KulsoEsOszlopok': 'Objetos/Casa/KulsoEsOszlopokFent_baseColor.jpeg',
        'KulsoEsOszlopokFent': 'Objetos/Casa/KulsoEsOszlopokFent_baseColor.jpeg',
        'KulsoFal': 'Objetos/Casa/KulsoFal_baseColor.jpeg',
        'Lepcso': 'Objetos/Casa/Lepcso_baseColor.jpeg',
        'Polc': 'Objetos/Casa/Polc_baseColor.jpeg',
        'Pult': 'Objetos/Casa/Pult_baseColor.jpeg',
        'Talaj': 'Objetos/Casa/Talaj_baseColor.jpeg',
        'Teto': 'Objetos/Casa/Teto_baseColor.jpeg',
    }
)

Cavalo = Objeto(
    obj_file='Objetos/Cavalo/cavalo.obj',
    textures_dict={
        'Body': 'Objetos/Cavalo/Body_albedo2.jpg',
        'Cannon_A.001': 'Objetos/Cavalo/Cannon_A_AlbedoTransparency.png',
        'Cannon_A_Metal.001': 'Objetos/Cavalo/Cannon_A_Metal_AlbedoTransparency.png',
        'Cannon_A_Wood.001': 'Objetos/Cavalo/Cannon_A_Wood_AlbedoTransparency.png',
        'Eyes': 'Objetos/Cavalo/Eye_brown_albedo.jpg',
        'Hair': 'Objetos/Cavalo/Neck_albedo.jpg',
        'Voorwagen_Harnass': 'Objetos/Cavalo/Voorwagen_Harnass_AlbedoTransparency.png',
        'Voorwagen_Metal': 'Objetos/Cavalo/Voorwagen_Metal_AlbedoTransparency.png',
        'Voorwagen_Wood': 'Objetos/Cavalo/Voorwagen_Wood_AlbedoTransparency.png',
    }
)

Ceu = Objeto(
    obj_file='Objetos/Ceu/ceu.obj',
    textures_dict={
        'Material.001': 'Objetos/Ceu/space-skybox-texture-mapping-cube-mapping-night-sky-24df7747449631f3f2a45fc630ae6ad0.png'
    }
)

Chao = Objeto(
    obj_file='Objetos/Chao/chao.obj',
    textures_dict={'Materiais': 'Objetos/Chao/4_Albedo.png'}
)

Fogueira = Objeto(
    obj_file='Objetos/Fogueira/fogueira.obj',
    textures_dict={
        'Coal': 'Objetos/Fogueira/Campfire_Coal_Diffuse.jpg',
        'Dast': 'Objetos/Fogueira/Campfire_Dast_Diffuse.jpg',
        'Wood': 'Objetos/Fogueira/Campfire_Wood_Diffuse.jpg'
    }
)

Jhon = Objeto(
    obj_file='Objetos/Jhon/jhon.obj',
    textures_dict={
        '.bmp': 'Objetos/Jhon/repeater_carbine01.png',
        'iv_drawable_material.bmp.001': 'Objetos/Jhon/melee_lasso01.png',
        'player_default_intro_alphakill_d1.bmp': 'Objetos/Jhon/player_default_intro_alphakill_d1.png'
    }
)

Lampiao = Objeto(
    obj_file='Objetos/Lampiao/lampiao.obj',
    textures_dict={
        'Glass_2.3': 'Objetos/Lampiao/Glass_2.3_baseColor.png',
        'Lamp_body': 'Objetos/Lampiao/Lamp_body_baseColor.png',
    }
)

Mapa = Objeto(
    obj_file='Objetos/Mapa/mapa.obj',
    textures_dict={'TreasureMap': 'Objetos/Mapa/TreasureMap_baseColor.png'}
)

Mesa = Objeto(
    obj_file='Objetos/Mesa/mesa.obj',
    textures_dict={'Material.002': 'Objetos/Mesa/Asztal_albedo.png'}
)

Nuvem = Objeto(
    obj_file='Objetos/Nuvem/nuvem.obj',
    textures_dict={}
)

Rifle = Objeto(
    obj_file='Objetos/Rifle/rifle.obj',
    textures_dict={
        'WoodenSmall': 'Objetos/Rifle/WoodenSmall_albedo.jpg',
        'WoodenBig': 'Objetos/Rifle/WoodenBig_albedo.jpg',
        'Pipe': 'Objetos/Rifle/WoodenBig_albedo.jpg',
        'Cock': 'Objetos/Rifle/WoodenBig_albedo.jpg',
        'MetalBody': 'Objetos/Rifle/WoodenBig_albedo.jpg',
        'MetalPart': 'Objetos/Rifle/WoodenBig_albedo.jpg',
        'PipeBetween': 'Objetos/Rifle/WoodenBig_albedo.jpg'
    }
)

# =========================
# TRANSFORMAÇÕES
# =========================

Barril.set_transformations(-20.5, 0, -16, 0.0, 0.0, 0.0, 2.30, 2.30, 2.30)
Cacto01.set_transformations(-56, 0.5, 4.5, 0.0, 0.0, 0.0, 2.20, 2.20, 2.20)
Cacto02.set_transformations(-38, 0.2, 36, 0.0, 0.0, 0.0, 3.51, 3.51, 3.51)
Cadeira.set_transformations(-24.5, -1.5, -1.5, 0.0, -90.0, 0.0, 1.40, 1.40, 1.40)
Cama.set_transformations(-32.38, 1, 8.5, 0, 0, 0, 0.3, 0.3, 0.3)
Casa.set_transformations(-28, 15, 0, 0, 0, 0, 3, 3, 3)
Cavalo.set_transformations(-49.625000, -1.750000, -16.500000, 0.0, 135.0, 0.0, 2.80, 2.80, 2.80)
Ceu.set_transformations(0, 0, 0, 0, 0, 0, 2, 2, 2)
Chao.set_transformations(0, 0, 0, 0, 0, 0, 40, 40, 40)
Fogueira.set_transformations(-21.5, -1, -37, -7.5, 0.0, 15.3, 0.15, 0.15, 0.15)
Jhon.set_transformations(-41.625000, 11.937500, -11.687500, 0.0, -140.0, 0.0, 7.60, 7.60, 7.60)
Lampiao.set_transformations(-23.5, 2.1875, -1.5, 0, 0, 0, 1, 1, 1)
Mapa.set_transformations(-20.75, 3.28125, 6.625, 32.5, -180.0, 0.0, 0.10, 0.10, 0.10)
Mesa.set_transformations(-23.5, -5, -1.5, 0, 0, 0, 2.5, 2.5, 2.5)
Nuvem.set_transformations(-22.671875, 7.000000, -26.000000, 0.0, 0.0, 0.0, 0.80, 1.20, 1.20)
Rifle.set_transformations(-26.140625, 4.375000, 14.843750, -5.0, 20.0, -15.0, 1.40, 1.40, 1.40)

for subobject in Nuvem.subobjects:
    subobject.set_color(0.64, 0.64, 0.64, 0.7)


# =========================
# APPEND & Set dos Centros dos Objetos
# =========================

Objects.extend([
    Barril, Cacto01, Cacto02, Cadeira, Cama, Casa,
    Cavalo, Ceu, Chao, Fogueira, Jhon, Lampiao,
    Mapa, Mesa, Nuvem, Rifle
])


#Nuvem.pivot_x, Nuvem.pivot_y, Nuvem.pivot_z = get_object_center(Nuvem)
Cavalo.pivot_x, Cavalo.pivot_y, Cavalo.pivot_z = get_object_center(Cavalo)
Rifle.pivot_x, Rifle.pivot_y, Rifle.pivot_z = get_object_center(Rifle)

#Nuvem.pivot_y = get_object_bottom_y(Nuvem)

wheels_index = [3, 4, 12, 13]

for i in wheels_index:
    sub = Cavalo.subobjects[i]
    cx, cy, cz = get_sub_object_center(sub)
    sub.pivot_x, sub.pivot_y, sub.pivot_z = cx, cy, cz

### Enviando os dados da CPU para a GPU, requisitando dois slots (buffers): um para os vértices e outro para as texturas.

In [301]:
buffer_VBO = glGenBuffers(2)

### Enviando coordenadas de vértices para a GPU

In [302]:
vertices = np.zeros(len(vertices_list), [("position", np.float32, 3)])
vertices['position'] = vertices_list


# Upload data
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[0])
glBufferData(GL_ARRAY_BUFFER, vertices.nbytes, vertices, GL_STATIC_DRAW)
stride = vertices.strides[0]
offset = ctypes.c_void_p(0)
loc_vertices = glGetAttribLocation(program, "position")
glEnableVertexAttribArray(loc_vertices)
glVertexAttribPointer(loc_vertices, 3, GL_FLOAT, False, stride, offset)

### Enviando coordenadas de textura para a GPU

In [303]:
textures = np.zeros(len(textures_coord_list), [("position", np.float32, 2)]) # duas coordenadas
textures['position'] = textures_coord_list


# Upload data
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[1])
glBufferData(GL_ARRAY_BUFFER, textures.nbytes, textures, GL_STATIC_DRAW)
stride = textures.strides[0]
offset = ctypes.c_void_p(0)
loc_texture_coord = glGetAttribLocation(program, "texture_coord")

glEnableVertexAttribArray(loc_texture_coord)
glVertexAttribPointer(loc_texture_coord, 2, GL_FLOAT, False, stride, offset)


### Eventos para modificar a posição da câmera.

* Usei as teclas A, S, D e W para movimentação no espaço tridimensional
* Usei a posição do mouse para "direcionar" a câmera

In [304]:
# Camera configs
cameraPos   = glm.vec3(-52, 15, -69.0)
cameraFront = glm.vec3(0.0, 0.0, 1.0)
cameraUp    = glm.vec3(0.0, 1.0, 0.0)

firstMouse = True
yaw   =  90.0
pitch =  0.0
lastX =  largura / 2.0
lastY =  altura / 2.0
fov   =  45.0

X_MIN = -201
X_MAX = 201
Y_MIN = 1
Y_MAX = 201
Z_MIN = -201
Z_MAX = 201

deltaTime = 0.0
lastFrame = 0.0


def key_event(window, key, scancode, action, mods):
    global cameraPos, cameraFront, cameraUp, polygonal_mode

    if key == glfw.KEY_ESCAPE and action == glfw.PRESS:
        glfw.set_window_should_close(window, True)

    # --- Camera: WASD ---
    cameraSpeed = 250 * deltaTime
    if key == glfw.KEY_W and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos += cameraSpeed * cameraFront
    if key == glfw.KEY_S and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos -= cameraSpeed * cameraFront
    if key == glfw.KEY_A and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos -= glm.normalize(glm.cross(cameraFront, cameraUp)) * cameraSpeed
    if key == glfw.KEY_D and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos += glm.normalize(glm.cross(cameraFront, cameraUp)) * cameraSpeed

    cameraPos.x = max(X_MIN, min(X_MAX, cameraPos.x))
    cameraPos.y = max(Y_MIN, min(Y_MAX, cameraPos.y))
    cameraPos.z = max(Z_MIN, min(Z_MAX, cameraPos.z))

    if key == glfw.KEY_P and action == glfw.PRESS:
        polygonal_mode = not polygonal_mode

    if key == glfw.KEY_F:
        for i in wheels_index:
            sub = Cavalo.subobjects[i]
            sub.angle_x += 1.5
    if key == glfw.KEY_G:
        for i in wheels_index:
            sub = Cavalo.subobjects[i]
            sub.angle_x -= 1.5

    if key == glfw.KEY_KP_ADD:
        Nuvem.scale_object(0.1, 0.2, 0.1)

        print(Nuvem.pivot_x, Nuvem.pivot_y, Nuvem.pivot_z)
    if key == glfw.KEY_KP_SUBTRACT: 
        Nuvem.scale_object(-0.1, -0.2, -0.1)

    if key == glfw.KEY_UP: Jhon.t_x += 0.3
    if key == glfw.KEY_DOWN: Jhon.t_x -= 0.3
    if key == glfw.KEY_RIGHT: Jhon.t_z += 0.3
    if key == glfw.KEY_LEFT: Jhon.t_z -= 0.3

def framebuffer_size_callback(window, largura, altura):
    glViewport(0, 0, largura, altura)


def mouse_callback(window, xpos, ypos):
    global cameraFront, lastX, lastY, firstMouse, yaw, pitch

    if firstMouse:
        lastX = xpos
        lastY = ypos
        firstMouse = False

    xoffset = (xpos - lastX) * 0.1
    yoffset = (lastY - ypos) * 0.1
    lastX = xpos
    lastY = ypos

    yaw   += xoffset
    pitch += yoffset
    pitch  = max(-89.0, min(89.0, pitch))

    front = glm.vec3(
        glm.cos(glm.radians(yaw)) * glm.cos(glm.radians(pitch)),
        glm.sin(glm.radians(pitch)),
        glm.sin(glm.radians(yaw)) * glm.cos(glm.radians(pitch))
    )
    cameraFront = glm.normalize(front)


def scroll_callback(window, xoffset, yoffset):
    global fov
    fov = max(1.0, min(45.0, fov - yoffset))


glfw.set_key_callback(window, key_event)
glfw.set_framebuffer_size_callback(window, framebuffer_size_callback)
glfw.set_cursor_pos_callback(window, mouse_callback)
glfw.set_scroll_callback(window, scroll_callback)
glfw.set_input_mode(window, glfw.CURSOR, glfw.CURSOR_DISABLED)

### Matrizes Model, View e Projection

In [305]:
def model(angle_x, angle_y, angle_z, t_x, t_y, t_z, s_x, s_y, s_z):
    mat = glm.mat4(1.0)

    mat = glm.translate(mat, glm.vec3(t_x, t_y, t_z))

    mat = glm.rotate(mat, glm.radians(angle_x), glm.vec3(1, 0, 0))
    mat = glm.rotate(mat, glm.radians(angle_y), glm.vec3(0, 1, 0))
    mat = glm.rotate(mat, glm.radians(angle_z), glm.vec3(0, 0, 1))

    mat = glm.scale(mat, glm.vec3(s_x, s_y, s_z))

    return np.array(mat)

def view():
    global cameraPos, cameraFront, cameraUp
    mat_view = glm.lookAt(cameraPos, cameraPos + cameraFront, cameraUp)
    mat_view = np.array(mat_view)
    return mat_view

def projection():
    global altura, largura
    # perspective parameters: fovy, aspect, near, far
    mat_projection = glm.perspective(glm.radians(fov), largura/altura, 0.1, 1000.0)

    
    mat_projection = np.array(mat_projection)    
    return mat_projection

### Exibindo a janela!


In [306]:
glfw.show_window(window)

### Loop principal da janela.

In [307]:
glEnable(GL_DEPTH_TEST)
polygonal_mode = False


while not glfw.window_should_close(window):

    currentFrame = glfw.get_time()
    deltaTime = currentFrame - lastFrame
    lastFrame = currentFrame

    glfw.poll_events() 
       
    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)
    
    glClearColor(1.0, 1.0, 1.0, 1.0)
    
    if polygonal_mode:
        glPolygonMode(GL_FRONT_AND_BACK,GL_LINE)
    else:
        glPolygonMode(GL_FRONT_AND_BACK,GL_FILL)

    for obj in Objects:
        obj.draw(program)
    
    mat_view = view()
    loc_view = glGetUniformLocation(program, "view")
    glUniformMatrix4fv(loc_view, 1, GL_TRUE, mat_view)

    mat_projection = projection()
    loc_projection = glGetUniformLocation(program, "projection")
    glUniformMatrix4fv(loc_projection, 1, GL_TRUE, mat_projection)    
    
    glfw.swap_buffers(window)

glfw.terminate()

0 0 0
0 0 0
0 0 0
0 0 0
0 0 0
0 0 0
0 0 0
0 0 0
0 0 0
0 0 0
0 0 0
0 0 0
0 0 0
0 0 0
0 0 0
